# NB4 · Explainability

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---

The substantive work here is assessing the limits of the explanation rather than the
correctness of the code.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
               'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'en'


In [ ]:
# Fixed cell. Rebuilds the output of the previous notebooks.
import pipeline as pl

state = pl.prepare(verbose=False)
model = state['model']
test = state['test']
features = state['features']
probability = state['probabilities']
y_test = state['y_test']
threshold = ev.threshold_for_sensitivity(y_test, probability, target=0.80)
print(f'Model and predictions ready. Operating threshold: {threshold:.3f}')


---

## Step 1 · Global explanation

The question is which features drive the model overall. Permutation importance measures
this in a model agnostic way, by observing how far performance falls when a feature is
shuffled.

In the result, the spread matters as much as the mean. On a small cohort the spread
across repeats exceeds the difference between neighbouring features. Where that happens
the ranking itself is unstable and cannot be presented as an ordering.


### Prompt 1

```
There is a fitted scikit-learn Pipeline named model and a DataFrame named test. The
list features holds the column names the model uses. The target column is target.

Write a single Python cell that computes permutation importance. Use roc_auc as the
scoring metric and 20 repeats.

CONTRACT
Produce a DataFrame named importance with the columns: feature, importance, spread
Sort it in descending order of importance and print the first fifteen rows.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 1


In [ ]:
checks.check_columns(importance, required=['feature', 'importance', 'spread'],
                     name='importance')

unstable = (importance['importance'] <= 2 * importance['spread']).sum()
print(f'\nFeatures with an unstable ranking: {unstable} / {len(importance)}')
print('Where this count is high, do not present the feature ordering as a finding.')


---

## Step 2 · Explaining single cases

Three cases are selected: A true positive, a false positive and a false negative.

The false positive is the one that matters. An explanation that makes a wrong prediction
look reasonable is the mechanism by which explanation launders error. Seeing it once
teaches more than learning the definition of the concept.

In logistic regression the contributions are exact rather than approximate. The log odds
is the sum of coefficient times feature value, so the contribution of each feature to a
single prediction can be read directly. That decomposition disappears with a non-linear
model, and a post hoc approximation such as SHAP becomes necessary.


### Prompt 2

```
I am working with the same model and test set. The array probability holds the
positive class probability for every row in the test set. The variable threshold
holds the decision threshold.

Write a single Python cell that:
1. Finds the row index of one true positive, one false positive and one false
   negative case.
2. Computes the feature contributions behind each of those predictions. The model is
   linear, so use coefficient times transformed feature value.
3. Prints the top eight contributions for each case alongside the actual feature
   values.

CONTRACT
Produce a dictionary named cases with the keys true_positive, false_positive and
false_negative, whose values are row indices. Where a case is not found, the value
must be None.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 2


In [ ]:
for name in ['true_positive', 'false_positive', 'false_negative']:
    found = cases.get(name)
    print(f'{name:<16} {"not found" if found is None else f"row {found}"}')


---

## Fixed analysis

The cells below are the fixed section of this notebook. They compute the contributions
exactly and show that the result is an identity: The contributions plus the intercept
give the log odds, whose sigmoid equals the probability the model returns. That equality
does not hold for an approximation.


In [ ]:
case = cases.get('false_positive') or cases.get('true_positive')
explanation = ex.explain_case(model, test[features], case, top=10**6)

from_contributions = 1 / (1 + np.exp(-explanation['log_odds']))
print(f"Probability from contributions : {from_contributions:.10f}")
print(f"Probability from the model     : {explanation['probability']:.10f}")
print(f"Difference                     : {abs(from_contributions - explanation['probability']):.2e}")


In [ ]:
for name in ['true_positive', 'false_positive', 'false_negative']:
    case = cases.get(name)
    if case is None:
        continue
    explanation = ex.explain_case(model, test[features], case, top=8)
    ex.plot_case(explanation, title=f"{name} · probability {explanation['probability']:.3f}")


## Critique of the explanation

Answer the following questions yourself, from the three plots.

Could one of the high contribution features be an artefact of how the record was made
rather than a clinical signal? The measurement count columns are candidates in this
cohort. A patient with many measurements in the first six hours may simply have been
treated as severe from the outset. That is a proxy for care intensity, not a clinical
finding.

Would a clinician reading the false positive explanation be persuaded that the
prediction was reasonable? If so, that is a problem rather than a success.

Which of the contributions could be read as a causal claim? A high contribution from a
low systolic pressure does not mean that raising the pressure would shorten the stay.

Put the same questions to the AI tool and compare its answers with your own. Asked to
criticise an explanation it produced itself, the tool is noticeably more critical than
when asked for a summary.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
